# Neural Networks Intro

**Topic:** Supervised Learning — Introduction to Deep Learning

> **A note on tools:** this notebook uses sklearn's `MLPClassifier` throughout,
> since it keeps the same `.fit()` / `.predict()` API you already know from
> every other algorithm in this folder. That's the right choice for a first
> "hello world" look at neural networks. It is not what real deep learning
> work uses — production teams build with Keras/TensorFlow or PyTorch, not
> sklearn. You'll meet those tools in `stretch/deep_learning/`.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.datasets import fetch_openml
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the structure of a feedforward neural network: layers, neurons, weights, and activation functions
- **Explain** how a neural network learns by propagating errors backward through layers (backpropagation)
- **Interpret** a network architecture diagram and a live digit-classifier, and connect depth/width to both model capacity and prediction confidence

> **Tip:** In the architecture widget, add layers and neurons and watch how fast the parameter count climbs. Later, you'll use that same kind of network to classify real handwritten digits from the MNIST dataset — the classic first dataset for learning neural networks.

---
## How we got here

Neural networks unify and generalize everything you have studied in this folder:

- **[supervised/05_logistic_regression.ipynb](05_logistic_regression.ipynb)** — a single neuron with a sigmoid activation is identical to logistic regression; a neural network is many logistic regression units stacked
- **[math/calculus/04_chain_rule.ipynb](../math/calculus/04_chain_rule.ipynb)** — backpropagation is the chain rule applied recursively through every layer; you cannot understand training without it
- This notebook is a conceptual preview. Full implementation is coming in the **stretch/deep_learning/** folder.

---
## Why this matters for data science

Neural networks are the foundation of everything in modern AI: image recognition, speech processing, large language models, code generation. You have spent this entire folder building up to this point: linear models, feature transformations, optimization, regularization, and ensemble ideas all appear inside neural networks.

The important message for this notebook is that neural networks are not magic. They are differentiable functions with parameters trained by gradient descent — the same ideas you have already learned, composed many times over. Their power comes from depth (many layers) and scale (many parameters), not from a fundamentally different kind of mathematics.

---
## Where it sits on the spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full discussion. On that spectrum, a neural network sits at the far end from everything else you have studied: **complexity 0.98 out of 1, interpretability close to 0 ("None")** — compare that to logistic regression's complexity of 0.10 and near-full interpretability.

That trade-off is the headline fact about neural networks. A logistic regression model can tell you exactly how much each pixel contributed to a prediction. A neural network with even one hidden layer cannot — the prediction emerges from thousands of weights interacting through nonlinear activations, and no single weight "means" anything on its own. You gain enormous flexibility to fit complex patterns (like handwritten digits) and lose the ability to explain any individual prediction in plain terms.

---
## Try it yourself

Use the sliders to change the network's depth (hidden layers) and width (neurons per layer). Watch two things: how the diagram grows, and how fast the parameter count climbs.

In [ ]:
layers_slider = widgets.IntSlider(min=1, max=4, step=1, value=2,
                                   description="Hidden layers:",
                                   style={"description_width": "120px"})
neurons_slider = widgets.IntSlider(min=4, max=128, step=4, value=32,
                                    description="Neurons/layer:",
                                    style={"description_width": "120px"})

out_arch = Output()

INPUT_SIZE = 784   # a flattened 28x28 MNIST image
OUTPUT_SIZE = 10   # digits 0-9
MAX_DOTS_PER_LAYER = 10  # cap drawn nodes per layer so the diagram stays readable

def param_count(n_layers, n_neurons):
    sizes = [INPUT_SIZE] + [n_neurons] * n_layers + [OUTPUT_SIZE]
    return sum(sizes[i] * sizes[i + 1] + sizes[i + 1] for i in range(len(sizes) - 1))

def render_arch(change=None):
    n_layers = layers_slider.value
    n_neurons = neurons_slider.value

    layer_ys = [list(np.linspace(-1, 1, min(8, MAX_DOTS_PER_LAYER)))]
    for _ in range(n_layers):
        layer_ys.append(list(np.linspace(-1, 1, min(n_neurons, MAX_DOTS_PER_LAYER))))
    layer_ys.append(list(np.linspace(-1, 1, OUTPUT_SIZE)))

    node_x, node_y = [], []
    for x, ys in enumerate(layer_ys):
        for y in ys:
            node_x.append(x)
            node_y.append(y)

    edge_x, edge_y = [], []
    for x in range(len(layer_ys) - 1):
        for a in layer_ys[x]:
            for b in layer_ys[x + 1]:
                edge_x += [x, x + 1, None]
                edge_y += [a, b, None]

    total_params = param_count(n_layers, n_neurons)
    baseline_params = param_count(0, n_neurons)
    tick_text = (["784 pixel<br>inputs"]
                 + [f"Hidden {i + 1}<br>({n_neurons} neurons)" for i in range(n_layers)]
                 + ["10 digit<br>outputs"])

    fig = go.Figure(
        data=[
            go.Scatter(x=edge_x, y=edge_y, mode="lines",
                       line=dict(color=PALETTE["muted"], width=0.4),
                       opacity=0.5, hoverinfo="skip", showlegend=False),
            go.Scatter(x=node_x, y=node_y, mode="markers",
                       marker=dict(size=13, color=PALETTE["primary"],
                                  line=dict(color="white", width=1)),
                       hoverinfo="skip", showlegend=False),
        ],
        layout=base_layout(
            title=f"{n_layers} hidden layer(s) x {n_neurons} neurons/layer — {total_params:,} parameters",
            xaxis_title="",
            yaxis_title="",
        ),
    )
    fig.update_xaxes(tickmode="array", tickvals=list(range(len(layer_ys))),
                     ticktext=tick_text, range=[-0.5, len(layer_ys) - 0.5])
    fig.update_yaxes(visible=False, range=[-1.3, 1.3])
    fig.update_layout(
        height=380, showlegend=False,
        annotations=[dict(
            text=(f"For comparison, a single-layer model (logistic regression) on the same "
                  f"784 pixels has {baseline_params:,} parameters."),
            xref="paper", yref="paper", x=0.5, y=-0.28, showarrow=False,
            font=dict(size=12, color=PALETTE["muted"]),
        )],
        margin=dict(b=90),
    )

    with out_arch:
        clear_output(wait=True)
        fig.show()

layers_slider.observe(render_arch, names="value")
neurons_slider.observe(render_arch, names="value")

display(VBox([HBox([layers_slider, neurons_slider]), out_arch]))
render_arch()

---
## What's happening?

As you slide the controls, notice two things: the parameter count grows *combinatorially*, not linearly — going from 1 hidden layer of 32 neurons to 2 hidden layers of 128 neurons multiplies the parameter count by roughly 4.6x, not the 2x or 4x you'd expect if depth and width scaled the parameter count linearly. And even a modest 2-layer, 64-neuron network already has more parameters than any tree-based model you have studied.

A neural network is a composition of simple functions. Each **neuron** computes a weighted sum of its inputs, adds a bias, and applies an **activation function** (sigmoid, ReLU, tanh) that introduces nonlinearity. Without activation functions, a stack of linear layers collapses to a single linear layer — you need the nonlinearity to gain representational power.

**Forward pass**: input flows through each layer, each neuron computes its output, and the final layer produces the prediction.

**Backpropagation**: the prediction error is computed at the output. The gradient of the error with respect to each weight is computed using the chain rule, flowing backward through the network. Each weight is then adjusted in the direction that reduces the error (gradient descent).

| Component | Plain English | Math equivalent |
|---|---|---|
| Neuron | A single logistic regression unit | $a = \sigma(\mathbf{w}^	op \mathbf{x} + b)$ |
| Layer | A set of neurons operating in parallel | Matrix multiply + bias + activation |
| Depth | Number of layers stacked | Number of function compositions |
| Width | Number of neurons per layer | Dimension of the hidden representation |
| Activation | Nonlinearity applied to each neuron | ReLU: $\max(0, z)$; sigmoid: $\sigma(z)$ |
| Backprop | Error flowing backward to update weights | Chain rule applied recursively |

---
## Real-world example: Classifying handwritten digits with a neural network

MNIST is the classic first dataset for neural networks: 70,000 handwritten digits (0-9), each a 28x28 pixel grayscale image flattened into 784 numbers. sklearn's MLPClassifier lets you train a feedforward neural network on it without leaving the familiar sklearn API — the same `.fit()` / `.predict()` interface as every algorithm you have used in this folder.

- **Notice:** Going from 1 hidden layer to 2 hidden layers, or from 32 to 128 neurons, measurably improves accuracy on digit recognition — this is a task where the extra capacity actually pays off
- **Notice:** Pixel values are scaled from their raw 0-255 range down to 0-1 before training; neural networks are sensitive to input scale, and unscaled pixel values would slow or destabilize training
- **Notice:** sklearn's MLPClassifier is for learning the concept; for production-scale image work (larger images, millions of examples) you would move to PyTorch or TensorFlow and add convolutional layers, which are far more efficient for images than flattening pixels

> **Discussion question:** Handwritten digits like 4 and 9, or 3 and 8, get confused more often than digits like 0 and 1. Why would a plain feedforward network — which has no idea a pixel is part of an "image" rather than just one of 784 unordered numbers — struggle more with visually similar digits?

### Neural network terminology

| Component | Plain English | sklearn/keras class |
|---|---|---|
| Feedforward network | Layers flow one direction: input to output | `MLPClassifier`, `MLPRegressor` |
| Convolutional network | Shares weights across spatial positions in images | `torch.nn.Conv2d` |
| Recurrent network | Has loops — processes sequences one step at a time | `torch.nn.LSTM` |
| Transformer | Attention-based; underlies GPT and BERT | `transformers.AutoModel` |
| Activation: ReLU | Zero below 0, linear above 0 — most common hidden layer activation | `activation='relu'` |
| Activation: Softmax | Converts output layer to probability distribution | `activation='softmax'` (output) |

In [ ]:
print("Loading MNIST (cached locally after the first run)...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_mnist_full, y_mnist_full = mnist.data, mnist.target.astype(int)

# Full MNIST is 70,000 images — subsample for a fast in-class fit.
# A few thousand examples is still enough to see real accuracy differences
# between architectures.
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_mnist_full), size=5000, replace=False)
X_sample, y_sample = X_mnist_full[sample_idx], y_mnist_full[sample_idx]

X_mnist_scaled = X_sample / 255.0  # raw pixels are 0-255; scale to 0-1

X_train_mnist, X_test_mnist, y_train_mnist, y_test_mnist = train_test_split(
    X_mnist_scaled, y_sample, test_size=0.2, random_state=42, stratify=y_sample
)

architectures = {
    "1 hidden (32)":     (32,),
    "1 hidden (64)":     (64,),
    "2 hidden (64x64)":  (64, 64),
    "2 hidden (128x64)": (128, 64),
}

names, accs = [], []
for label, arch in architectures.items():
    m = MLPClassifier(hidden_layer_sizes=arch, max_iter=200,
                      random_state=42, early_stopping=True)
    m.fit(X_train_mnist, y_train_mnist)
    acc = accuracy_score(y_test_mnist, m.predict(X_test_mnist))
    names.append(label)
    accs.append(acc)
    print(f"{label:20s}  Accuracy: {acc:.4f}")

fig = go.Figure(data=[
    go.Bar(x=names, y=accs,
           marker_color=PALETTE["primary"],
           text=[f"{a:.4f}" for a in accs], textposition="outside"),
], layout=base_layout(
    title="Neural Network on MNIST: Accuracy by Architecture",
    xaxis_title="Architecture",
    yaxis_title="Accuracy",
))
fig.update_layout(yaxis=dict(range=[0.8, 1.0]), showlegend=False)
fig.show()

---
### Try it yourself: watch it predict real digits

Pick an architecture and click for a new set of sample digits. Watch the predicted label and confidence under each image — and notice which digits the smallest network gets wrong that the larger ones get right.

In [ ]:
ARCH_PRESETS = {
    "Tiny — 1 layer x 16 neurons":    (16,),
    "Small — 1 layer x 32 neurons":   (32,),
    "Medium — 2 layers x 64 neurons": (64, 64),
    "Large — 2 layers x 128 neurons": (128, 128),
}

preset_dd = widgets.Dropdown(options=list(ARCH_PRESETS.keys()),
                              value="Medium — 2 layers x 64 neurons",
                              description="Architecture:",
                              style={"description_width": "100px"})
shuffle_btn = widgets.Button(description="New sample digits")
out_pred = Output()

_model_cache = {}
_sample_state = {"idx": None}
pred_rng = np.random.default_rng(7)

def get_model(preset_name):
    if preset_name not in _model_cache:
        arch = ARCH_PRESETS[preset_name]
        m = MLPClassifier(hidden_layer_sizes=arch, max_iter=200,
                          random_state=42, early_stopping=True)
        m.fit(X_train_mnist, y_train_mnist)
        _model_cache[preset_name] = m
    return _model_cache[preset_name]

def render_pred(change=None):
    model = get_model(preset_dd.value)
    idx = _sample_state["idx"]
    imgs = X_test_mnist[idx].reshape(-1, 28, 28)
    true = y_test_mnist[idx]
    preds = model.predict(X_test_mnist[idx])
    probs = model.predict_proba(X_test_mnist[idx])
    conf = probs.max(axis=1)

    fig = make_subplots(rows=2, cols=4,
                         subplot_titles=[f"True {t} · Pred {p}<br>{c:.0%} confidence"
                                         for t, p, c in zip(true, preds, conf)],
                         horizontal_spacing=0.08, vertical_spacing=0.18)
    fig.update_annotations(font_size=11)
    for i, img in enumerate(imgs):
        r, c = divmod(i, 4)
        fig.add_trace(go.Heatmap(z=img[::-1], colorscale="Greys",
                                  reversescale=True, showscale=False),
                      row=r + 1, col=c + 1)
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    fig.update_layout(
        height=420, width=760,
        title=f"{preset_dd.value} — accuracy on these 8 digits: {(preds == true).mean():.0%}",
    )

    with out_pred:
        clear_output(wait=True)
        fig.show()

def new_sample(_=None):
    _sample_state["idx"] = pred_rng.choice(len(X_test_mnist), size=8, replace=False)
    render_pred()

preset_dd.observe(render_pred, names="value")
shuffle_btn.on_click(new_sample)

display(VBox([HBox([preset_dd, shuffle_btn]), out_pred]))
new_sample()

---
## When to use it / When NOT to use it

| Use it when | Do NOT use it when |
|---|---|
| Your data is images, audio, text, or another high-dimensional signal without hand-built features | Your data is a modest-sized tabular dataset — tree-based models usually match or beat a neural network with far less tuning |
| You have a large amount of training data (thousands to millions of examples) | You have a small dataset — neural networks overfit fast without enough data to constrain all those parameters |
| Interpretability is not a hard requirement | You must explain individual predictions to a regulator, clinician, or customer |
| You have time and infrastructure to tune architecture, learning rate, and regularization | You need a fast, low-maintenance baseline — start with a simpler model first |

> **A neural network is many logistic regression units stacked in layers — depth and nonlinear activations give it the capacity to learn any function, but that capacity requires careful regularization and much more data than classical supervised algorithms.**

---
*Coming up in stretch/deep_learning/: convolutional networks, recurrent networks, transformers, and PyTorch from scratch*

---
*Next up: 17 — Model Comparison, the capstone that benchmarks every algorithm head-to-head*